# 喜び・悲しみ・怒りなど 8 種類の感情の強さを推定する (WRIME)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/emotion_ja.ipynb)

NLP クックブック 12-27 ／ テキスト分類

ポジティブかネガティブかだけでなく、投稿に表れた喜び・悲しみ・怒りなどの感情を、種類ごとに強さとして推定したい。書き手本人と読み手とで感じ方がどう違うかも見る。

In [1]:
!pip install -q fugashi unidic-lite   # 後半で使う BERT の日本語トークナイザーに必要

In [2]:
import numpy as np
import pandas as pd

# WRIME Ver.1 (梶原ら, NAACL 2021, https://github.com/ids-cv/wrime): SNS の投稿 43,200 件に、Plutchik の 8 感情の強度
# (0:無 1:弱 2:中 3:強) を書き手本人 1 人と読み手 3 人が付けたもの。CC BY-NC-ND 4.0 (非商用・改変禁止)
url = "https://raw.githubusercontent.com/ids-cv/wrime/master/wrime-ver1.tsv"
df = pd.read_csv(url, sep="\t")
EMO = ["Joy", "Sadness", "Anticipation", "Surprise", "Anger", "Fear", "Disgust", "Trust"]
JA = ["喜び", "悲しみ", "期待", "驚き", "怒り", "恐れ", "嫌悪", "信頼"]
W = [f"Writer_{e}" for e in EMO]   # 書き手 (主観)
# 読み手 (客観) は 3 人の強度の平均 (実数)。TSV の「Avg. Readers_*」列は平均を四捨五入した整数なので使わない
for e in EMO:
    df[f"Readers_{e}"] = df[[f"Reader{i}_{e}" for i in (1, 2, 3)]].mean(axis=1)
R = [f"Readers_{e}" for e in EMO]
train, test = df[df["Train/Dev/Test"] == "train"], df[df["Train/Dev/Test"] == "test"]
print(len(df), "件", df["Train/Dev/Test"].value_counts().to_dict())
print("書き手の人数:", df.groupby("Train/Dev/Test")["UserID"].nunique().to_dict(),
      " 学習とテストで書き手が重ならない:", set(train["UserID"]).isdisjoint(test["UserID"]))

def mean_over_readers(f):   # 読み手 3 人 (または 3 組) について平均する
    return lambda e: np.mean([f(e, i) for i in (1, 2, 3)])

pd.DataFrame({
    "書き手 ≥1 の割合": [(df[f"Writer_{e}"] >= 1).mean() for e in EMO],
    "読み手 1 人 ≥1 の割合": [mean_over_readers(lambda e, i: (df[f"Reader{i}_{e}"] >= 1).mean())(e) for e in EMO],
    "書き手と読み手 1 人の相関": [mean_over_readers(lambda e, i: df[f"Writer_{e}"].corr(df[f"Reader{i}_{e}"]))(e) for e in EMO],
    "読み手どうしの相関": [mean_over_readers(lambda e, i: df[f"Reader{i}_{e}"].corr(df[f"Reader{i % 3 + 1}_{e}"]))(e) for e in EMO],
}, index=JA).round(3)

43200 件 {'train': 40000, 'test': 2000, 'dev': 1200}
書き手の人数: {'dev': 10, 'test': 10, 'train': 60}  学習とテストで書き手が重ならない: True


,書き手 ≥1 の割合,読み手 1 人 ≥1 の割合,書き手と読み手 1 人の相関,読み手どうしの相関
喜び,0.403,0.262,0.568,0.641
悲しみ,0.354,0.212,0.449,0.533
期待,0.374,0.287,0.410,0.581
驚き,0.288,0.213,0.376,0.466
怒り,0.129,0.028,0.401,0.627
恐れ,0.176,0.185,0.355,0.453
嫌悪,0.209,0.140,0.383,0.466
信頼,0.217,0.051,0.175,0.216


In [3]:
# 書き手は強い感情 (3) を付けたのに、読み手 3 人が全員 0 を付けた投稿
# 本文の出典: WRIME Ver.1 (梶原ら, NAACL 2021, https://github.com/ids-cv/wrime)、
# CC BY-NC-ND 4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/)。先頭 60 字の抜粋
for e, ja in [("Anger", "怒り"), ("Trust", "信頼")]:
    strong = df[df[f"Writer_{e}"] == 3]
    hit = strong[(strong[[f"Reader{i}_{e}" for i in (1, 2, 3)]] == 0).all(axis=1)]
    print(f"{ja}: 書き手が 3 を付けた {len(strong)} 件のうち、読み手 3 人とも 0 は {len(hit)} 件")
    for s in hit["Sentence"].sample(2, random_state=0):
        print("   ", s[:60])

怒り: 書き手が 3 を付けた 1488 件のうち、読み手 3 人とも 0 は 784 件
    酷い話だ･･
    暑すぎて死ぬ
信頼: 書き手が 3 を付けた 1947 件のうち、読み手 3 人とも 0 は 1220 件
    朝七時前のビール。業の極み。
    せっかく痩せてきたのに･･･。また太るなぁw


In [4]:
from sentence_transformers import SentenceTransformer

# 学習データ 40,000 件から 10,000 件を使う。Colab の CPU で埋め込みの計算に約 3〜5 分
sub = train.sample(10000, random_state=0)
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
encode = lambda texts: embedder.encode(["query: " + t for t in texts], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
X_train, X_test = encode(sub["Sentence"].tolist()), encode(test["Sentence"].tolist())
print(X_train.shape, X_test.shape)

(10000, 384) (2000, 384)


In [5]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import roc_auc_score

# 8 感情の強度をまとめて回帰する (出力 8 次元のリッジ回帰。正則化の強さは学習データ内の一個抜き交差検証で選ぶ)
alphas = [0.1, 0.3, 1, 3, 10]
ridge_w = RidgeCV(alphas=alphas).fit(X_train, sub[W])   # 書き手の強度を予測
ridge_r = RidgeCV(alphas=alphas).fit(X_train, sub[R])   # 読み手 3 人の平均強度を予測
pred_emb = {"書き手": ridge_w.predict(X_test), "読み手": ridge_r.predict(X_test)}

def evaluate(pred):
    """感情ごとに、強度との相関 r と、「強度 ≥1 (その感情あり)」を当てる ROC-AUC を求める"""
    rows = []
    for who, cols in [("書き手", W), ("読み手", R)]:
        for i, ja in enumerate(JA):
            y = test[cols[i]].to_numpy()
            rows.append({"感情": ja, "立場": who, "正例数": int((y >= 1).sum()),
                         "相関 r": np.corrcoef(y, pred[who][:, i])[0, 1], "AUC": roc_auc_score(y >= 1, pred[who][:, i])})
    return pd.DataFrame(rows).pivot(index="感情", columns="立場").loc[JA]

res_emb = evaluate(pred_emb)
res_emb.round(3)

正例数        相関 r           AUC       
立場   書き手  読み手    書き手    読み手    書き手    読み手
感情                                       
喜び   843  454  0.537  0.659  0.787  0.886
悲しみ  734  288  0.384  0.488  0.739  0.823
期待   915  553  0.360  0.582  0.700  0.847
驚き   511  184  0.295  0.443  0.658  0.805
怒り   345   37  0.408  0.241  0.792  0.840
恐れ   375  265  0.321  0.501  0.713  0.798
嫌悪   430  195  0.382  0.448  0.754  0.824
信頼   362   29  0.204  0.417  0.643  0.897

In [6]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# WRIME Ver.1 (NC-ND) の train で、書き手 8 + 読み手 3 人の平均 8 の計 16 個の強度を回帰するよう学習した BERT。
# モデルカードの表示は CC BY-SA 3.0
name = "patrickramos/bert-base-japanese-v2-wrime-fine-tune"
tokenizer = AutoTokenizer.from_pretrained(name)
bert = AutoModelForSequenceClassification.from_pretrained(name).eval()

def bert_predict(texts, bs=64):
    order = np.argsort([len(t) for t in texts])   # 長さの近い文をまとめてパディングを減らす
    out = np.zeros((len(texts), bert.config.num_labels), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            idx = order[i:i + bs]
            batch = tokenizer([texts[j] for j in idx], padding=True, truncation=True, max_length=128, return_tensors="pt")
            out[idx] = bert(**batch).logits.numpy()
    return out

# Colab の CPU で、テスト 2,000 件の予測に約 3〜5 分 (モデルのダウンロード約 450MB は別)
logits = bert_predict(test["Sentence"].tolist())
pred_bert = {"書き手": logits[:, :8], "読み手": logits[:, 8:]}
res_bert = evaluate(pred_bert)
pd.concat({"埋め込み + リッジ": res_emb["相関 r"], "BERT (WRIME で学習)": res_bert["相関 r"]}, axis=1).round(3)

埋め込み + リッジ        BERT (WRIME で学習)       
立場         書き手    読み手              書き手    読み手
感情                                           
喜び       0.537  0.659            0.645  0.817
悲しみ      0.384  0.488            0.494  0.663
期待       0.360  0.582            0.438  0.769
驚き       0.295  0.443            0.449  0.672
怒り       0.408  0.241            0.528  0.564
恐れ       0.321  0.501            0.409  0.640
嫌悪       0.382  0.448            0.483  0.652
信頼       0.204  0.417            0.266  0.544

In [7]:
# 8 感情の平均の相関と、その 95% 信頼区間。テストは 10 人の投稿なので、投稿ではなく書き手を単位に
# リサンプリングする (同じ人の投稿どうしは似ているので、投稿単位だと区間が狭く出すぎる)
rng = np.random.default_rng(0)
users = test["UserID"].to_numpy()
rows_of = {u: np.flatnonzero(users == u) for u in np.unique(users)}
boots = [np.concatenate([rows_of[u] for u in rng.choice(list(rows_of), len(rows_of))]) for _ in range(300)]

def mean_r(cols, pred, ix):
    y, p = test[cols].to_numpy()[ix], pred[ix]
    return np.mean([np.corrcoef(y[:, i], p[:, i])[0, 1] for i in range(8)])

for model_name, pred in [("埋め込み + リッジ", pred_emb), ("BERT (WRIME で学習)", pred_bert)]:
    for who, cols in [("書き手", W), ("読み手", R)]:
        b = [mean_r(cols, pred[who], ix) for ix in boots]
        print(f"{model_name:16} {who}: 平均 r = {mean_r(cols, pred[who], slice(None)):.3f}"
              f"  (書き手単位のブートストラップ 95% CI {np.percentile(b, 2.5):.3f}〜{np.percentile(b, 97.5):.3f})")

埋め込み + リッジ       書き手: 平均 r = 0.361  (書き手単位のブートストラップ 95% CI 0.319〜0.403)
埋め込み + リッジ       読み手: 平均 r = 0.473  (書き手単位のブートストラップ 95% CI 0.443〜0.494)
BERT (WRIME で学習) 書き手: 平均 r = 0.464  (書き手単位のブートストラップ 95% CI 0.408〜0.514)
BERT (WRIME で学習) 読み手: 平均 r = 0.665  (書き手単位のブートストラップ 95% CI 0.628〜0.685)


In [8]:
texts = ["やっと試験に合格した!今夜はお祝いだ", "財布を落とした…最悪", "明日からの旅行が楽しみすぎて眠れない",
         "また電車が遅れてる。いい加減にしてほしい", "夜中に変な物音がして怖い"]
logits = bert_predict(texts)
for who, sl in [("書き手", slice(0, 8)), ("読み手", slice(8, 16))]:
    print(f"\nBERT の予測 ({who})")
    print(pd.DataFrame(logits[:, sl], columns=JA, index=[t[:14] for t in texts]).round(1).to_string())


BERT の予測 (書き手)
                 喜び  悲しみ   期待   驚き   怒り   恐れ   嫌悪   信頼
やっと試験に合格した!今夜は  2.9  0.1  1.4  0.1  0.0  0.0  0.0  0.8
財布を落とした…最悪      0.1  2.4  0.0  0.2  0.7  0.7  1.4 -0.0
明日からの旅行が楽しみすぎて  2.4  0.1  2.6 -0.0  0.0  0.1  0.1  0.9
また電車が遅れてる。いい加減  0.1  1.1  0.0  0.2  2.2  0.5  2.2 -0.0
夜中に変な物音がして怖い   -0.0  0.3  0.0  0.9 -0.1  2.4  0.5 -0.1

BERT の予測 (読み手)
                 喜び  悲しみ   期待   驚き   怒り   恐れ   嫌悪   信頼
やっと試験に合格した!今夜は  2.5 -0.0  0.9 -0.0 -0.0 -0.0 -0.0  0.1
財布を落とした…最悪      0.1  2.0  0.0 -0.0  0.2  0.5  1.2 -0.0
明日からの旅行が楽しみすぎて  1.5  0.1  2.2 -0.0  0.0  0.1  0.2  0.1
また電車が遅れてる。いい加減  0.1  0.7  0.1  0.1  1.3  0.3  2.0 -0.0
夜中に変な物音がして怖い   -0.0 -0.1 -0.0  0.4 -0.1  2.6  0.2  0.0


- 多くの感情で、書き手は読み手より多くの感情を報告する（恐れだけは逆）。特に怒りと信頼は、強度 1 以上を付けた割合が書き手で 12.9% と 21.7%、読み手 1 人あたりでは 2.8% と 5.1% しかない。書き手が信頼を 3（強）と付けた 1,947 件のうち 1,220 件は、読み手 3 人とも 0 だった。本人が感じていても文面に出ていない感情は、テキストからは推定しにくい。
- 書き手と読み手 1 人の相関は、どの感情でも読み手どうしの相関より低い。差が大きいのは怒り（0.401 と 0.627）と期待（0.410 と 0.581）。信頼は読み手どうしでも 0.216 と一致しにくい。
- 8 感情の平均の相関は、どちらのモデルでも読み手のほうが書き手より高かった（埋め込み + リッジで 0.473 と 0.361、BERT で 0.665 と 0.464。書き手単位のブートストラップ信頼区間は重ならない）。文面から伝わる感情のほうが推定しやすいことに加え、読み手は 3 人の平均なので 1 人分のラベルよりノイズが少ないことも効いている。用途に応じて、どちらのラベルで学習・評価するかを決める（「読んだ人にどう伝わるか」なら読み手、「本人の状態」なら書き手）。
- ほとんど 0 ばかりの目標では、相関 r が低く出やすい。読み手の怒りはテストで強度 1 以上が 37 件しかなく、埋め込み + リッジの相関は 0.241 と書き手 (0.408) より低いが、「その感情あり」を当てる AUC は 0.840 と書き手 (0.792) より高い。出現の少ない感情は、正例数を確かめ、相関と AUC の両方を見る。
- 強度を回帰しておけば、しきい値を決めて「この感情あり / なし」のマルチラベル分類にも使える。しきい値の決め方や評価は、この章の「1 つの文書に複数のラベルを付ける（マルチラベル分類）」「クラスの偏ったデータを扱う（クラス重み・しきい値調整）」と同じ。回帰なので、予測が 0 未満になることもある。
- WRIME で学習済みの BERT は、16 項目すべてで埋め込み + リッジより相関が高かった。ただし BERT は学習データ 40,000 件全部でモデル全体を微調整しており、条件は対等ではない。また、このモデルの学習コードは学習中の評価にテスト分割を使っている。最良のエポックを選ぶ設定ではなく最終エポックのモデルだが、テストを見ながら設定を決めた可能性は否定できないので、BERT の数値はやや楽観的に見る。
- WRIME の学習・検証・テストは書き手で分かれている（テストの 10 人は学習に出てこない）。同じ人の投稿が学習とテストの両方に入ると、書き手の癖を覚えて評価が甘くなる。信頼区間も、投稿ではなく書き手を単位にリサンプリングして求めた。ブートストラップによる信頼区間の求め方は評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照。
- WRIME は CC BY-NC-ND 4.0（非商用・改変禁止）。BERT のモデルは CC BY-SA 3.0 と表示されているが、WRIME で微調整したものなので、商用で使う場合は WRIME の条件も含めて確認が必要。ポジネガだけでよければ、この章の「日本語の感情分析（ポジネガ判定）をする」を使う。